# [교육 자료] PINN을 활용한 임의 형상(DXF/사용자 정의) 경계 조건 전자기장 해석

본 실습 자료는 학생들이 HFSS 또는 AutoCAD 등의 툴에서 설계 후 내보낸 **DXF 도면 파일**이나, 직접 설계한 **임의의 형상(예: Bow-tie 안테나, Patch 안테나)**을 물리 정보 신경망(PINN)의 경계 조건(Boundary Condition)으로 적용하여 주변의 전위 및 전기장 분포를 해석하는 과정을 다룬다.

## 1. 전자기 해석에서 임의 경계 조건의 중요성
안테나 및 RF 설계에서는 전계의 효율적인 방사나 전하의 분산을 위해 다채로운 기하학적 형상(Geometry)을 사용한다. 기존의 FDTD, FEM 등 수치해석적 기법은 형상이 복잡해질수록 미세한 격자(Mesh)를 나눌 때 엄청난 양의 연산 비용이 소모된다.
반면, PINN은 도메인에 콜로케이션 포인트(Collocation Points)를 정의하여 학습하므로, 복잡한 설계 도면을 좌표 기반의 데이터 집합으로 손쉽게 매핑하여 라플라스 방정식과 경계 조건을 최적화할 수 있는 강력한 장점을 지니고 있다.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import time
import ipywidgets as widgets
from IPython.display import display

# 재현성을 위한 난수 시드 고정
torch.manual_seed(42)
np.random.seed(42)


## 2. 드래그 앤 드롭 DXF 파일 업로드 위젯
아래 위젯을 실행하여 AutoCAD 또는 HFSS에서 Export한 2D `.dxf` 파일을 마우스로 드래그 앤 드롭(Drag & Drop)하여 업로드할 수 있다.
업로드된 파일이 없을 경우 기본 수학적 모델인 **Bow-tie 안테나** 또는 **Patch 안테나** 형상을 사용한다.

> **[DXF 작성 팁]** DXF 도면을 그릴 때 도체의 성질에 따라 레이어 이름을 아래와 같이 설정하면 자동으로 전압이 할당된다.
> *   `POS` 포함 레이어: 전위 $V = +1.0\text{V}$ (양극 도체)
> *   `NEG` 또는 `GND` 포함 레이어: 전위 $V = -1.0\text{V}$ 또는 $0.0\text{V}$ (음극/접지)
> *   `OUTER` 또는 `BOUND` 포함 레이어: 영역의 외부 가상 경계 ($V = 0.0\text{V}$)


In [ ]:
# DXF 파일 업로드 위젯 표시 (마우스 드래그 앤 드롭 업로드 지원)
dxf_uploader = widgets.FileUpload(
    accept='.dxf',
    multiple=False,
    description='DXF 파일 업로드'
)
print("여기에 DXF 파일을 클릭하여 선택하거나 마우스로 드래그 앤 드롭 하세요:")
display(dxf_uploader)


## 3. 사용자 정의 경계 조건 데이터셋 생성 (DXF 파싱 및 기하학적 모델링)

업로드된 DXF 파일에서 도체의 경계점($X_u$)을 추출하고 자유 공간 포인트($X_f$)를 필터링하여 콜로케이션 포인트 셋을 구성한다.
업로드된 파일이 없거나 오류 발생 시 지정된 `geometry_type` ('bowtie' 또는 'patch')에 맞는 수학적 형상을 자동으로 생성한다.


In [ ]:
# ==========================================
# 1. 임의 형상 경계 조건 생성 및 콜로케이션 포인트 매핑
# ==========================================

# 업로드된 파일 파싱 처리
uploaded_dxf_path = None
if dxf_uploader.value:
    value = dxf_uploader.value
    content = None
    filename = "uploaded_temp.dxf"
    
    if isinstance(value, (list, tuple)): # ipywidgets 8.x+
        file_info = value[0]
        content = file_info['content']
        filename = file_info['name']
    elif isinstance(value, dict): # ipywidgets 7.x
        filename = list(value.keys())[0]
        content = value[filename]['content']
        
    if content is not None:
        if isinstance(content, memoryview):
            content = content.tobytes()
        with open(filename, "wb") as f:
            f.write(content)
        uploaded_dxf_path = filename
        print(f"[알림] 업로드된 DXF 파일 '{filename}'을 감지했습니다. 이 파일을 경계 조건으로 사용합니다.")

# (A) DXF 도면 레이어별 파서 함수
def load_dxf_by_layer(dxf_path):
    try:
        import ezdxf
    except ImportError:
        print("[경고] DXF 도면을 파싱하려면 'ezdxf' 라이브러리가 필요합니다. ('pip install ezdxf' 필요)")
        return None, None, None
        
    doc = ezdxf.readfile(dxf_path)
    msp = doc.modelspace()
    
    pos_pts = []
    neg_pts = []
    outer_pts = []
    
    def extract_pts_from_entity(entity):
        pts = []
        if entity.dxftype() == 'LINE':
            start = entity.dxf.start
            end = entity.dxf.end
            for t in np.linspace(0, 1, 15):
                pts.append([start.x + t*(end.x - start.x), start.y + t*(end.y - start.y)])
        elif entity.dxftype() == 'LWPOLYLINE':
            poly_pts = list(entity.get_points('xy'))
            for idx in range(len(poly_pts) - 1):
                p1, p2 = np.array(poly_pts[idx]), np.array(poly_pts[idx+1])
                for t in np.linspace(0, 1, 10):
                    pts.append(p1 + t*(p2 - p1))
        return pts

    for entity in msp:
        layer_name = entity.dxf.layer.upper()
        extracted = extract_pts_from_entity(entity)
        if not extracted:
            continue
            
        if 'POS' in layer_name:
            pos_pts.extend(extracted)
        elif 'NEG' in layer_name or 'GND' in layer_name:
            neg_pts.extend(extracted)
        elif 'OUTER' in layer_name or 'BOUND' in layer_name:
            outer_pts.extend(extracted)
        else:
            pos_pts.extend(extracted) # 레이어가 애매할 경우 기본 전극(POS) 처리
            
    return (np.array(pos_pts) if pos_pts else None, 
            np.array(neg_pts) if neg_pts else None, 
            np.array(outer_pts) if outer_pts else None)

# (B) 수학적 모델링 설정
geometry_type = 'bowtie' # 업로드된 DXF가 없을 시 활성화할 기본 수학 모델 ('bowtie' 또는 'patch')

N_u_pts = 200 # 각 도체 전극당 샘플링 포인트 수
N_f = 3000    # 자유 공간 PDE 콜로케이션 포인트 수

# 삼각형 내부 균일 샘플링
def sample_triangle_interior(v1, v2, v3, num_pts):
    v1, v2, v3 = np.array(v1), np.array(v2), np.array(v3)
    r1 = np.random.rand(num_pts, 1)
    r2 = np.random.rand(num_pts, 1)
    sqrt_r1 = np.sqrt(r1)
    pts = (1 - sqrt_r1) * v1 + sqrt_r1 * (1 - r2) * v2 + sqrt_r1 * r2 * v3
    return pts

# 삼각형 테두리 샘플링
def sample_triangle_boundary(v1, v2, v3, num_pts):
    v1, v2, v3 = np.array(v1), np.array(v2), np.array(v3)
    pts_per_edge = num_pts // 3
    edge1 = v1 + np.random.rand(pts_per_edge, 1) * (v2 - v1)
    edge2 = v2 + np.random.rand(pts_per_edge, 1) * (v3 - v2)
    edge3 = v3 + np.random.rand(pts_per_edge, 1) * (v1 - v3)
    return np.vstack((edge1, edge2, edge3))

# 도메인 가상 외곽선 생성 (V = 0.0V)
x_l, y_l = -np.ones((50, 1)), np.random.uniform(-1.0, 1.0, (50, 1))
x_r, y_r = np.ones((50, 1)), np.random.uniform(-1.0, 1.0, (50, 1))
y_b, x_b = -np.ones((50, 1)), np.random.uniform(-1.0, 1.0, (50, 1))
y_t, x_t = np.ones((50, 1)), np.random.uniform(-1.0, 1.0, (50, 1))
X_outer = np.vstack((np.hstack((x_l, y_l)), np.hstack((x_r, y_r)), np.hstack((x_b, y_b)), np.hstack((x_t, y_t))))
V_outer = np.zeros((X_outer.shape[0], 1))

# 자유 공간 난수 포인트 전체 생성 (나중에 도체 부분 필터링)
x_f_rand = np.random.uniform(-1.0, 1.0, (N_f * 4, 1))
y_f_rand = np.random.uniform(-1.0, 1.0, (N_f * 4, 1))
X_f_all = np.hstack((x_f_rand, y_f_rand))

if uploaded_dxf_path is not None:
    X_pos_dxf, X_neg_dxf, X_outer_dxf = load_dxf_by_layer(uploaded_dxf_path)
    if X_pos_dxf is not None:
        X_pos = X_pos_dxf
        X_neg = X_neg_dxf if X_neg_dxf is not None else np.array([[0.0, -0.8]]) # 음극이 없을 시 대략적인 접지 좌표
        if X_outer_dxf is not None: X_outer = X_outer_dxf
        
        V_pos = np.ones_like(X_pos[:, :1]) * 1.0
        V_neg = np.zeros_like(X_neg[:, :1]) # 임의로 0V로 접지
        V_outer = np.zeros_like(X_outer[:, :1])
        
        # 업로드된 임의 도형과의 빠른 벡터 거리 필터링 (도체 경계 0.06 반경 내의 영역은 자유공간에서 제외)
        dists_pos = np.min(np.sum((X_f_all[:, np.newaxis, :] - X_pos[np.newaxis, :, :])**2, axis=2), axis=1)
        dists_neg = np.min(np.sum((X_f_all[:, np.newaxis, :] - X_neg[np.newaxis, :, :])**2, axis=2), axis=1)
        mask = (dists_pos > 0.06**2) & (dists_neg > 0.06**2)
        X_f_np = X_f_all[mask][:N_f]
        geometry_type = 'dxf'
    else:
        uploaded_dxf_path = None # 파싱 실패 시 수학 모델로 롤백

if uploaded_dxf_path is None:
    # 업로드 파일이 없을 시 로직
    if geometry_type == 'bowtie':
        v_pos1, v_pos2, v_pos3 = [0.0, 0.05], [0.4, 0.4], [-0.4, 0.4]
        v_neg1, v_neg2, v_neg3 = [0.0, -0.05], [0.4, -0.4], [-0.4, -0.4]
        
        pos_boundary = sample_triangle_boundary(v_pos1, v_pos2, v_pos3, N_u_pts // 2)
        pos_interior = sample_triangle_interior(v_pos1, v_pos2, v_pos3, N_u_pts // 2)
        X_pos = np.vstack((pos_boundary, pos_interior))
        V_pos = np.ones_like(X_pos[:, :1]) * 1.0
        
        neg_boundary = sample_triangle_boundary(v_neg1, v_neg2, v_neg3, N_u_pts // 2)
        neg_interior = sample_triangle_interior(v_neg1, v_neg2, v_neg3, N_u_pts // 2)
        X_neg = np.vstack((neg_boundary, neg_interior))
        V_neg = np.ones_like(X_neg[:, :1]) * -1.0

    elif geometry_type == 'patch':
        x_patch = np.random.uniform(-0.3, 0.3, (N_u_pts - 50, 1))
        y_patch = np.random.uniform(0.2, 0.5, (N_u_pts - 50, 1))
        x_feed = np.random.uniform(-0.05, 0.05, (50, 1))
        y_feed = np.random.uniform(-0.8, 0.2, (50, 1))
        X_pos = np.vstack((np.hstack((x_patch, y_patch)), np.hstack((x_feed, y_feed))))
        V_pos = np.ones_like(X_pos[:, :1]) * 1.0
        
        x_gnd = np.random.uniform(-1.0, 1.0, (N_u_pts, 1))
        y_gnd = -0.8 * np.ones_like(x_gnd)
        X_neg = np.hstack((x_gnd, y_gnd))
        V_neg = np.zeros_like(X_neg[:, :1]) * 0.0

    # 수학 도형 내부에 속한 포인트 필터링
    if geometry_type == 'bowtie':
        def is_in_triangle(p, a, b, c):
            def cross_product(p1, p2, p3):
                return (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])
            d1 = cross_product(p, a, b)
            d2 = cross_product(p, b, c)
            d3 = cross_product(p, c, a)
            has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
            has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)
            return not (has_neg and has_pos)
            
        mask = []
        for p in X_f_all:
            in_pos = is_in_triangle(p, v_pos1, v_pos2, v_pos3)
            in_neg = is_in_triangle(p, v_neg1, v_neg2, v_neg3)
            mask.append(not (in_pos or in_neg))
        X_f_np = X_f_all[mask][:N_f]

    elif geometry_type == 'patch':
        mask = []
        for p in X_f_all:
            in_patch = (-0.3 <= p[0] <= 0.3) and (0.2 <= p[1] <= 0.5)
            in_feed = (-0.05 <= p[0] <= 0.05) and (-0.8 <= p[1] <= 0.2)
            mask.append(not (in_patch or in_feed))
        X_f_np = X_f_all[mask][:N_f]

X_f = torch.FloatTensor(X_f_np)
X_f.requires_grad_(True)

X_u = torch.FloatTensor(np.vstack((X_pos, X_neg, X_outer)))
V_u = torch.FloatTensor(np.vstack((V_pos, V_neg, V_outer)))

# 생성된 콜로케이션 포인트 시각화
plt.figure(figsize=(7, 7))
plt.scatter(X_f[:, 0].detach().numpy(), X_f[:, 1].detach().numpy(), c='lightgray', s=3, label='Free Space (PDE Points)')
plt.scatter(X_pos[:, 0], X_pos[:, 1], c='red', s=10, label='Positive Pole')
plt.scatter(X_neg[:, 0], X_neg[:, 1], c='blue', s=10, label='Negative/Ground Pole')
plt.scatter(X_outer[:, 0], X_outer[:, 1], c='black', s=5, label='Outer Boundary')
plt.title(f'Sampled Collocation Points (Geometry: {geometry_type.upper()})')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.show()


## 4. 하이퍼파라미터 설정 및 다이내믹 모델 구축

모델 아키텍처 및 훈련 환경을 설정한다.
은닉층의 구성과 훈련 주기를 조율하여 복잡한 기하학적 형태에 전자기 법칙이 적절히 수렴되는지 파악할 수 있다.


In [ ]:
# ==========================================
# 2. 하이퍼파라미터 설정 및 모델 정의
# ==========================================

learning_rate = 1e-3
epochs = 3000
hidden_layers = [64, 64, 64, 64]
activation_func = nn.Tanh()

print(f"설정 학습률: {learning_rate}")
print(f"목표 에포크: {epochs}")
print(f"은닉층 형태: {hidden_layers}")
print(f"활성화 함수: {activation_func.__class__.__name__}")

# 모델 가시화 함수
def draw_neural_net(input_size, hidden_layers, output_size):
    vis_hidden = [min(h, 8) for h in hidden_layers]
    layer_sizes = [input_size] + vis_hidden + [output_size]
    
    fig = plt.figure(figsize=(10, 6))
    ax = fig.gca()
    ax.axis('off')
    
    left, right, bottom, top = 0.1, 0.9, 0.1, 0.9
    v_spacing = (top - bottom) / float(max(layer_sizes))
    h_spacing = (right - left) / float(len(layer_sizes) - 1)
    
    for n, layer_size in enumerate(layer_sizes):
        layer_top = v_spacing * (layer_size - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size):
            circle = plt.Circle((n * h_spacing + left, layer_top - m * v_spacing), v_spacing / 5.,
                                color='w', ec='b', zorder=4)
            ax.add_artist(circle)
            if m == layer_size - 1 and hidden_layers[0] > 8 and n > 0 and n < len(layer_sizes)-1:
                ax.text(n * h_spacing + left, layer_top - (m+1.5) * v_spacing, f'\n...\n({hidden_layers[n-1]})', ha='center', va='center', fontsize=10)
            
    for n, (layer_size_a, layer_size_b) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        layer_top_a = v_spacing * (layer_size_a - 1) / 2. + (top + bottom) / 2.
        layer_top_b = v_spacing * (layer_size_b - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size_a):
            for o in range(layer_size_b):
                line = plt.Line2D([n * h_spacing + left, (n + 1) * h_spacing + left],
                                  [layer_top_a - m * v_spacing, layer_top_b - o * v_spacing], c='k', alpha=0.1)
                ax.add_artist(line)
    
    plt.title("Dynamic PINN Net Topology")
    plt.show()

draw_neural_net(2, hidden_layers, 1)

# 다이내믹 모델 클래스 선언
class DynamicPINN(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size, activation):
        super(DynamicPINN, self).__init__()
        self.layers = nn.ModuleList()
        self.activation = activation
        
        in_size = input_size
        for h_size in hidden_layers:
            self.layers.append(nn.Linear(in_size, h_size))
            in_size = h_size
            
        self.output_layer = nn.Linear(in_size, output_size)
        
    def forward(self, x):
        out = x
        for layer in self.layers:
            out = self.activation(layer(out))
        out = self.output_layer(out)
        return out, None

# 인스턴스 초기화
model = DynamicPINN(input_size=2, hidden_layers=hidden_layers, output_size=1, activation=activation_func)
print(model)


## 5. 물리 잔차(PDE Loss) 계산 정의

도체가 없는 자유 공간 영역에서는 정상 상태 라플라스 방정식(Poisson's Equation에서 소스 전하가 없는 꼴)을 충족해야 한다:

$$\Delta V = V_{xx} + V_{yy} = 0$$

이를 위해 `torch.autograd.grad`를 이용하여 각 입력 공간 좌표 $(x, y)$에 대한 모델 출력 전위 $V$의 2차 도함수를 역전파가 가능한 형태로 미분 연산하여 손실 값을 산출한다.


In [ ]:
# ==========================================
# 3. 물리 잔차(PDE Loss) 계산 함수 정의
# ==========================================

def calc_pde_loss(model_net, x_points):
    pred, _ = model_net(x_points)
    
    # 1차 편미분 (dV/dx, dV/dy)
    grad_V = torch.autograd.grad(pred, x_points, grad_outputs=torch.ones_like(pred), create_graph=True)[0]
    dV_dx = grad_V[:, 0:1]
    dV_dy = grad_V[:, 1:2]
    
    # 2차 편미분 (d2V/dx2, d2V/dy2)
    grad_dV_dx = torch.autograd.grad(dV_dx, x_points, grad_outputs=torch.ones_like(dV_dx), create_graph=True)[0]
    d2V_dx2 = grad_dV_dx[:, 0:1]
    
    grad_dV_dy = torch.autograd.grad(dV_dy, x_points, grad_outputs=torch.ones_like(dV_dy), create_graph=True)[0]
    d2V_dy2 = grad_dV_dy[:, 1:2]
    
    # 라플라시안 Residual 오차 계산
    laplacian = d2V_dx2 + d2V_dy2
    loss_pde = torch.mean(laplacian ** 2)
    return loss_pde


## 6. 모델 최적화 및 훈련 과정 진행

이제 Adam 옵티마이저를 사용해 전체 손실 함수 $Total\ Loss = 100.0 \times Loss_{BC} + Loss_{PDE}$를 최적화하는 훈련 루프를 구동한다.
경계 조건($Loss_{BC}$)이 전극의 모양을 유지하게 만들어 주므로 보통 물리 방정식($Loss_{PDE}$)보다 높은 가중치를 주어 학습 안정성을 제어한다.


In [ ]:
# ==========================================
# 4. 신경망 모델 반복 훈련 루프
# ==========================================

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_history = []
start_time = time.time()

print(f"--- {geometry_type.upper()} 안테나 전자기 해석 모델 PINN 학습 시작 ---")
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # 1. 경계 조건 전위 수렴 (Loss_BC)
    u_pred_bc, _ = model(X_u)
    Loss_BC = nn.MSELoss()(u_pred_bc, V_u)
    
    # 2. 자유 공간 라플라스 수식 충족 (Loss_PDE)
    Loss_PDE = calc_pde_loss(model, X_f)
    
    # 3. 전체 가중 손실 산출 및 역전파
    loss = 100.0 * Loss_BC + Loss_PDE
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if (epoch + 1) % 300 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:4d}/{epochs}], Loss: {loss.item():.6e} (BC_MSE: {Loss_BC.item():.6e}, PDE_MSE: {Loss_PDE.item():.6e})")

print(f"학습 완료! (소요 시간: {time.time() - start_time:.2f}초)")

# 훈련 손실 시각화
plt.figure(figsize=(7, 4))
plt.plot(loss_history, color='purple', label='Total Loss')
plt.yscale('log')
plt.xlabel('Epochs')
plt.ylabel('Loss (Log Scale)')
plt.title('PINN Electromagnetics Convergence Curve')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()


## 7. 학습 결과 해석 및 정전기장 벡터 시각화

학습이 끝난 모델을 가지고 전체 영역에서의 전위 분포 $V(x, y)$와
맥스웰 방정식에 기인한 전계 관계식 $\vec{E} = -\nabla V$를 미분 연산으로 계산하여 전계의 세기(Magnitude)와 전기력선의 방향(Field Vector)을 도식화한다.
복잡한 형태(Bow-tie의 뾰족한 끝, Patch의 평평한 에지면 등)를 도체가 가지면서 발생하는 전자기장의 왜곡과 강도를 분석할 수 있다.


In [ ]:
# ==========================================
# 5. 전위 및 전계 성분 계산 및 시각화
# ==========================================

# 그리드 포인트 생성
x_grid = np.linspace(-1.0, 1.0, 100)
y_grid = np.linspace(-1.0, 1.0, 100)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
grid_tensor = torch.FloatTensor(np.c_[X_mesh.ravel(), Y_mesh.ravel()])
grid_tensor.requires_grad = True

# 모델을 활용한 전위 추론
V_pred, _ = model(grid_tensor)

# 공간 기울기(Gradient) 계산을 통해 전기장 도출 (E = -grad V)
grad_V = torch.autograd.grad(V_pred, grid_tensor, grad_outputs=torch.ones_like(V_pred), create_graph=False)[0]

Ex = -grad_V[:, 0:1].detach().numpy().reshape(X_mesh.shape)
Ey = -grad_V[:, 1:2].detach().numpy().reshape(X_mesh.shape)
E_magnitude = np.sqrt(Ex**2 + Ey**2)
V_pred_np = V_pred.detach().numpy().reshape(X_mesh.shape)

# 대형 시각화 패널 작성
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# 1. 2D 전위 (Potential V) 분포도
contour_v = ax1.contourf(X_mesh, Y_mesh, V_pred_np, levels=50, cmap='viridis')
fig.colorbar(contour_v, ax=ax1, label='Potential V (V)')

# 안테나 전극 아웃라인 표시
if geometry_type == 'bowtie':
    poly_p = plt.Polygon([[0.0, 0.05], [0.4, 0.4], [-0.4, 0.4]], closed=True, fill=False, edgecolor='red', linewidth=2.5, label='Positive (+1V)')
    poly_n = plt.Polygon([[0.0, -0.05], [0.4, -0.4], [-0.4, -0.4]], closed=True, fill=False, edgecolor='blue', linewidth=2.5, label='Negative (-1V)')
    ax1.add_patch(poly_p)
    ax1.add_patch(poly_n)
elif geometry_type == 'patch':
    rect_patch = plt.Rectangle((-0.3, 0.2), 0.6, 0.3, fill=False, edgecolor='red', linewidth=2.5, label='Patch (+1V)')
    rect_feed = plt.Rectangle((-0.05, -0.8), 0.1, 1.0, fill=False, edgecolor='orange', linewidth=2.0, label='Feedline')
    line_gnd = plt.Line2D([-1.0, 1.0], [-0.8, -0.8], color='blue', linewidth=2.5, label='Ground (0V)')
    ax1.add_patch(rect_patch)
    ax1.add_patch(rect_feed)
    ax1.add_line(line_gnd)
else:
    # DXF 모드일 경우 경계 포인트를 산점도로 함께 표시
    ax1.scatter(X_pos[:, 0], X_pos[:, 1], color='red', s=2, label='Pos Conductor')
    ax1.scatter(X_neg[:, 0], X_neg[:, 1], color='blue', s=2, label='Neg Conductor')

ax1.set_title("Predicted Electric Potential V(x,y)")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.legend(loc='upper right')

# 2. 2D 전기장 세기 및 벡터 화살표 (E-field Vectors & Magnitude)
contour_e = ax2.contourf(X_mesh, Y_mesh, E_magnitude, levels=50, cmap='plasma')
fig.colorbar(contour_e, ax=ax2, label='E-field Magnitude (V/m)')

# 전기장 방향 벡터장 (Quiver) 드로잉
sub = 5
ax2.quiver(X_mesh[::sub, ::sub], Y_mesh[::sub, ::sub], 
           Ex[::sub, ::sub] / (E_magnitude[::sub, ::sub] + 1e-8), 
           Ey[::sub, ::sub] / (E_magnitude[::sub, ::sub] + 1e-8), 
           color='white', alpha=0.5, scale=25)

if geometry_type == 'bowtie':
    poly_p_e = plt.Polygon([[0.0, 0.05], [0.4, 0.4], [-0.4, 0.4]], closed=True, fill=False, edgecolor='red', linewidth=2.5)
    poly_n_e = plt.Polygon([[0.0, -0.05], [0.4, -0.4], [-0.4, -0.4]], closed=True, fill=False, edgecolor='blue', linewidth=2.5)
    ax2.add_patch(poly_p_e)
    ax2.add_patch(poly_n_e)
elif geometry_type == 'patch':
    rect_patch_e = plt.Rectangle((-0.3, 0.2), 0.6, 0.3, fill=False, edgecolor='red', linewidth=2.5)
    rect_feed_e = plt.Rectangle((-0.05, -0.8), 0.1, 1.0, fill=False, edgecolor='orange', linewidth=2.0)
    line_gnd_e = plt.Line2D([-1.0, 1.0], [-0.8, -0.8], color='blue', linewidth=2.5)
    ax2.add_patch(rect_patch_e)
    ax2.add_patch(rect_feed_e)
    ax2.add_line(line_gnd_e)
else:
    # DXF 모드일 경우 경계 포인트를 산점도로 함께 표시
    ax2.scatter(X_pos[:, 0], X_pos[:, 1], color='red', s=2)
    ax2.scatter(X_neg[:, 0], X_neg[:, 1], color='blue', s=2)

ax2.set_title("E-field Magnitude & Directional Vectors")
ax2.set_xlabel("x")
ax2.set_ylabel("y")

plt.tight_layout()
plt.show()
